# Street and active travel infrastructure
## Mexicali Urban Liveability Index — `WP07_street_infrastructure`

**Lead:** TBC
**Indicators assigned:** 4
**Schema version:** 1.0.0

Pedestrian infrastructure, sidewalk availability, street lighting and cycling infrastructure, as segment-level attributes of the street network.

Compute on the street network first (per segment), then aggregate to areas with length weighting. Use the same network edges as the GHSCI pipeline so results are directly comparable.

> New to this project? Work through
> [`00_overview_and_schema.ipynb`](00_overview_and_schema.ipynb)
> first — it carries one indicator end to end. Then read
> [`docs/analyst_guide.md`](../docs/analyst_guide.md).

## How to work through this notebook

For each indicator assigned to you, in this order:

1. **Read the brief.** It reproduces everything the team already
   recorded in the workbook — the draft rationale, the article the
   indicator was adapted from, candidate data sources, and the open
   questions colleagues raised. Do not retype any of it; it is
   already in your metadata stub.
2. **Write the causal pathway sentence** (guide §2.1) and find
   **independent health evidence** for it (§2.2). Do this *before*
   looking for data. Fill in `meta['rationale']`.
3. **Find and document the data** (§3): citation, URL, date
   retrieved, licence, and whether it reaches Condesa.
4. **Compute** at the finest scale your data genuinely support.
   Produce a `DataFrame` with `geo_id` and `value`.
5. **Harmonise** with `uli.harmonise(...)`, label with
   `uli.label(...)`, and **deliver** with
   `uli.write_indicator(...)`.
6. **Look at the map.** Most errors are obvious in ten seconds and
   invisible in a table.

`uli.write_indicator` validates first and refuses to publish a
failing deliverable. While you are still iterating, pass
`allow_failure=True` to write a draft anyway.

Full guidance: [`docs/analyst_guide.md`](../docs/analyst_guide.md).
Schema: [`schema/ULI_output_schema.md`](../schema/ULI_output_schema.md).

## Framing the indicator against health evidence

Every indicator must be justified by evidence of a **meaningful
health or wellbeing benefit**, independent of the liveability
article it was adapted from. Those articles establish that an
indicator is used; they rarely establish that it matters.

Complete this sentence before you compute anything:

> *[what I measure]* changes *[a mechanism]*, which changes *[a
> behaviour or exposure]*, which affects *[a health outcome]*.

For most indicators in this project the behaviour is **walking for
transport**, **walking or recreation in public space**, or
**social contact** — and the exposure is **heat**, **air
pollution** or **injury risk**. Say which, using the vocabulary in
`uli.vocab.HEALTH_PATHWAYS`.

Prefer meta-analyses and systematic reviews, then reputable
guidance (WHO, UN-Habitat, PAHO, Secretaría de Salud), then cohort
studies and natural experiments. Record the **effect size with its
uncertainty**.

**If the evidence supports a different threshold from the one the
workbook proposes, use the evidence-based threshold** and say so in
`threshold_justification`. That is explicitly what the project
wants.

**Mexicali is arid and extremely hot.** Most of this literature
comes from temperate cities. Where the transfer is doubtful — for
example, distance-based walkability thresholds in a city where
summer maxima exceed 45 °C and shade rather than distance is the
binding constraint — record it in `rationale.arid_context`. That is
a contribution, not a caveat.

## When several workbook rows are really one indicator

The workbook harvested indicators article by article, so a single
construct sometimes appears as several rows seen through different
lenses or over different time periods. Air quality is the clearest
case:

| Row | What it is | Lens | Time basis |
|---|---|---|---|
| #292 Air quality | the index value itself | `quality` | `annual_mean` |
| #8 Good air quality | that value against a standard | `quality` | `threshold_share` |
| #293 Days with good air quality | how often the standard is met | `quantity` | `threshold_compliance_days` |
| #173 Days PM2.5 over WHO | the same, for one pollutant | `quantity` | `threshold_exceedance_days` |

These are not four indicators — they are one construct measured
four ways, and computing them separately would mean four
inconsistent methods and four sets of data documentation.

Deliver them as a **measure family**: give every measure the same
`measure_family` slug, and distinguish them with `temporal_basis`
(see `uli.vocab.TEMPORAL_BASES`) and `threshold`. They can still
live under separate workbook ids — the family slug is what tells
the index step, and Reimagina Urbana, that they belong together.

```python
for meta in (meta_292, meta_8, meta_293, meta_173):
    for measure in meta['measures']:
        measure['measure_family'] = 'air_quality'
    meta['data_sources'] = SHARED_SOURCES   # one method, one source
```

The same pattern applies to mean summer temperature versus days
above a comfort threshold (WP02), and to flood extent versus annual
average days of flooding (WP05).

## Condesa coverage is a requirement, not a nicety

The Condesa new development in south-east Mexicali is a project
focus area, and it defeats the usual assumptions:

- about **20%** of it falls outside the previously configured
  study region boundary;
- only **44%** of its area is covered by census manzana polygons,
  so a **manzana-native calculation reaches 33 of the 40
  fraccionamientos, while a `grid_100m`-native one reaches all
  40**;
- it is platted and roaded (43 km of street network in OpenStreetMap
  across 27 of the 40 fraccionamientos) but essentially unbuilt —
  **zero destinations**, and satellite-derived population products
  see almost nobody there.

**One thing is your decision: the native scale.** If your data
allow it, compute on the 100 m grid. That is the difference between
reaching all of Condesa and quietly missing a fifth of it.

Everything else is handled downstream. Population denominators,
the 2030 occupancy scenario and population-weighted exposure
statistics are a reporting-step concern (`uli.exposure`), decided
once for the whole project rather than by each analyst. Urban
fabric and exposure measures — land cover, air quality, heat,
hazards, street infrastructure — are properties of *place*, and
should be computed as such; who lives there is applied later.

Two things to record, though:

- `data_sources[].condesa_coverage` — whether your **source**
  reaches Condesa. Satellite imagery and OSM generally do; a 2020
  census variable or a household survey generally does not.
- `method.condesa_treatment` — what you did about it. Where a
  source does not reach Condesa, mark those rows `no_data` rather
  than omitting them.

The validator treats poor Condesa coverage as an **error**.

---
## Setup

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath('..'))

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import uli

# Identify yourself once; it is copied into every deliverable.
ANALYST = {
    'name': 'TODO: your name',
    'email': None,
    'institution': None,
}

print(f'ULI schema version {uli.SCHEMA_VERSION}')
print(f'Reference geographies available: {uli.geography.available()}')

        WORK_PACKAGE = 'WP07_street_infrastructure'
        NOTEBOOK = 'notebooks/07_street_infrastructure.ipynb'

---
## Your indicators (4)

Each has a brief reproducing what the workbook records,
then three working cells: documentation, calculation,
delivery.

### 61 — Cycling infrastructure

`cycling_infrastructure` · *Mobility & Transport · Active Travel · Cycling infrastructure · Cycling infrastructure*

- **Lenses to deliver:** accessibility
- **Draft rationale (rewrite this):** Dedicated cycling infrastructure encourages sustainable, eco-friendly transportation and promotes physical health through regular active mobility.
- **Adapted from:** Mekuria et al., 2017, 'Improving Liveability Using Green and Active Modes'; Stehlin, 2014, 'Regulating inclusion: Spatial form, social process, and the normalization of cycling practice in the USA'; Ruggeri, 2013, 'From Transit Stop to Urbanity Node'; Whitney et al., 2020, 'Livable streets and global competitiveness: A survey of Mexico City'
- **Effect reported there:** Compliance rate: 85.7% of evaluated liveability indices in the studied communities were related to public transport and green space availability.
- **Candidate data sources:** IMIP: https://www.mexicali.gob.mx/sitioimip/geovisor/geovisor/?url=pimus&access_token=
- **Team notes:** CH: This sounds like a suite of indicators/ ER: Agreed, we could use the cycling indicators we have generating for this

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_61) at any time to list what is
# still outstanding.
meta_61 = uli.metadata_stub(61, analyst=ANALYST)

# meta_61['rationale']['statement'] = """..."""
# meta_61['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_61['rationale']['arid_context'] = '...'
# meta_61['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_61['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_61)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_61 = 'grid_100m'
METHOD_61 = 'population_weighted_mean'

native_61 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_61 = uli.harmonise(
    native_61,
    native_scale=NATIVE_SCALE_61,
    method=METHOD_61,
)
results_61 = uli.label(
    harmonised_61,
    meta_61,
    measure_id='cycling_infrastructure__accessibility',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_61, meta_61))
# uli.write_indicator(results_61, meta_61)

### 75 — Sidewalk availability

`sidewalk_availability` · *Safety · Road Safety · Pedestrian Infrastructure · Sidewalk availability*

- **Lenses to deliver:** quantity
- **Draft rationale (rewrite this):** Adequate footpath provision is a key building block of liveable neighbourhoods, directly supporting the physical health and activity levels of residents.
- **Adapted from:** Hooper et al., 2015, 'The building blocks of a liveable neighbourhood: Identifying the key performance indicators for walking'
- **Effect reported there:** Association reported; no effect size provided.
- **Methods used in the literature:** ['Design Analysis']
- **Open questions raised:** ER: Only footpaths may not be feasible but we do have data for sidewalk availability

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_75) at any time to list what is
# still outstanding.
meta_75 = uli.metadata_stub(75, analyst=ANALYST)

# meta_75['rationale']['statement'] = """..."""
# meta_75['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_75['rationale']['arid_context'] = '...'
# meta_75['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_75['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_75)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_75 = 'grid_100m'
METHOD_75 = 'population_weighted_mean'

native_75 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_75 = uli.harmonise(
    native_75,
    native_scale=NATIVE_SCALE_75,
    method=METHOD_75,
)
results_75 = uli.label(
    harmonised_75,
    meta_75,
    measure_id='sidewalk_availability__quantity',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_75, meta_75))
# uli.write_indicator(results_75, meta_75)

### 167 — Pedestrian infrastructure

`pedestrian_infrastructure` · *Mobility & Transport · Active Travel · Pedestrian Infrastructure · Pedestrian infrastructure*

- **Lenses to deliver:** accessibility
- **Draft rationale (rewrite this):** Proper sidewalks and crosswalks are critical for safe walkability, encouraging residents to engage in health-promoting physical activities.
- **Adapted from:** Baobeid et al., 2021, 'Walkability and its relationships with health, sustainability, and liveability'; Geller, 2003, 'Smart growth: A prescription for livable cities'; Furlan, 2015, 'Liveability and Social Capital in West Bay'
- **Effect reported there:** Proximity target: All essential amenities and recreational facilities in the proposed design are located within a 15-minute walk.
- **Candidate data sources:** Inventario Nacional de Vivienda

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_167) at any time to list what is
# still outstanding.
meta_167 = uli.metadata_stub(167, analyst=ANALYST)

# meta_167['rationale']['statement'] = """..."""
# meta_167['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_167['rationale']['arid_context'] = '...'
# meta_167['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_167['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_167)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_167 = 'grid_100m'
METHOD_167 = 'population_weighted_mean'

native_167 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_167 = uli.harmonise(
    native_167,
    native_scale=NATIVE_SCALE_167,
    method=METHOD_167,
)
results_167 = uli.label(
    harmonised_167,
    meta_167,
    measure_id='pedestrian_infrastructure__accessibility',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_167, meta_167))
# uli.write_indicator(results_167, meta_167)

### 365 — Street light infrastructure

`street_light_infrastructure` · *Safety · Road Safety · Pedestrian Infrastructure · Street light infrastructure*

- **Lenses to deliver:** quality
- **Draft rationale (rewrite this):** Well-lit pedestrian networks are essential for social sustainability, creating a safe environment that encourages nighttime use of public space.
- **Adapted from:** Ahmed, 2012, 'Urban social sustainability: A study of the Emirati local communities in Al Ain'
- **Effect reported there:** No health outcome assessed.
- **Pragmatic approach agreed:** Proportion of road segments with presence of public lightning
- **Methods used in the literature:** ['Field observation']
- **Candidate data sources:** Inventario Nacional de Vivienda
- **Open questions raised:** I think its good to measure if we can, but the evidence is complicated (e.g. real/perceived safety vs carbon emissions and 'night sky' preservation from light pollution) https://pmc.ncbi.nlm.nih.gov/articles/PMC4509526/#s0075
- **Feasibility flag:** yes, data believed available

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_365) at any time to list what is
# still outstanding.
meta_365 = uli.metadata_stub(365, analyst=ANALYST)

# meta_365['rationale']['statement'] = """..."""
# meta_365['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_365['rationale']['arid_context'] = '...'
# meta_365['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_365['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_365)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_365 = 'grid_100m'
METHOD_365 = 'population_weighted_mean'

native_365 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_365 = uli.harmonise(
    native_365,
    native_scale=NATIVE_SCALE_365,
    method=METHOD_365,
)
results_365 = uli.label(
    harmonised_365,
    meta_365,
    measure_id='street_light_infrastructure__quality',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_365, meta_365))
# uli.write_indicator(results_365, meta_365)

---
## Check what this work package has delivered

In [ ]:
delivered, catalogue = uli.collect()
if len(catalogue):
    display(catalogue)
    print(delivered.groupby(['indicator_code', 'geo_level']).size())
else:
    print('Nothing delivered yet.')

In [ ]:
# Sanity-check a delivered measure on a map before you call it done.
# MEASURE = 'your_indicator_code__quantity'
# LEVEL = 'manzana'
# units = uli.geography.load(LEVEL).merge(
#     delivered.query('measure_id == @MEASURE and geo_level == @LEVEL'),
#     on='geo_id', how='left')
# ax = units.plot(column='value', legend=True, figsize=(11, 8),
#                 missing_kwds={'color': 'lightgrey'})
# condesa = uli.geography.load('condesa_fraccionamiento')
# condesa.boundary.plot(ax=ax, color='red', linewidth=1)
# ax.set_title(MEASURE)
# ax.set_axis_off()